In [1]:
import os
import random
import gc

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline


def set_seed(seed=2021):
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(2021)



In [2]:
train_path = "../input/ventilator-pressure-prediction/train.csv"
test_path = "../input/ventilator-pressure-prediction/test.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# Round continuous features to increase exact‑match coverage
df_train["time_step_r"] = df_train["time_step"].round(2)
df_train["u_in_r"] = df_train["u_in"].round().astype(int)

df_test["time_step_r"] = df_test["time_step"].round(2)
df_test["u_in_r"] = df_test["u_in"].round().astype(int)

feature_cols = ["R", "C", "time_step", "u_in", "u_out"]

X_train = df_train[feature_cols].values
y_train = df_train["pressure"].values

X_test = df_test[feature_cols].values

# Ridge model kept as a fallback (unchanged)
pipeline = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=3, include_bias=False),
    Ridge(alpha=0.001, random_state=2021),
)

pipeline.fit(X_train, y_train)
ridge_pred = pipeline.predict(X_test)

# ----- Exact‑mean lookup using rounded continuous features -----
exact_means = df_train.groupby(["R", "C", "time_step_r", "u_in_r", "u_out"])[
    "pressure"
].mean()

mid_means = df_train.groupby(["R", "C", "u_out"])["pressure"].mean()

# Merge predictions step‑by‑step
test_pred = df_test.merge(
    exact_means.rename("pressure_exact"),
    left_on=["R", "C", "time_step_r", "u_in_r", "u_out"],
    right_index=True,
    how="left",
)

test_pred = test_pred.merge(
    mid_means.rename("pressure_mid"),
    left_on=["R", "C", "u_out"],
    right_index=True,
    how="left",
)

global_mean = df_train["pressure"].mean()
ridge_series = pd.Series(ridge_pred, index=test_pred.index)

final_pred = (
    test_pred["pressure_exact"]
    .fillna(test_pred["pressure_mid"])
    .fillna(ridge_series)
    .fillna(global_mean)
)

final_pred = final_pred.clip(lower=0, upper=80).values



In [3]:
submission = pd.DataFrame({"id": df_test["id"], "pressure": final_pred})

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission file '{submission_path}' created with shape:", submission.shape)

Submission file 'submission.csv' created with shape: (603600, 2)
